# AntMaze preference pairs: jumping in preferred vs. non-preferred segments

This notebook analyzes the D4RL AntMaze **preference** datasets in `data/antmaze/` whose filenames contain `nt` (the `*_pref_nt.hdf5` files).

Each file stores preference *pairs*. For pair $i$ there are two trajectory segments:

* segment 1 — `states[i]`  (shape `(T, 29)`)
* segment 2 — `states_2[i]` (shape `(T, 29)`)

and a one-hot `labels[i]` of length 2 indicating which segment is **preferred**:
`labels[i] == [1, 0]` → segment 1 preferred, `labels[i] == [0, 1]` → segment 2 preferred.

### Defining a “jump”
The 29-dim AntMaze observation is the ant's `qpos` (15) followed by `qvel` (14). Index **2** is the **torso z-height** (the global x, y are kept at indices 0 and 1). The resting torso height is ≈ 0.5; during a jump the torso rises substantially (values reach ≈ 1.36).

A segment **contains a jump** if the torso z-height exceeds the threshold `JUMP_Z` at **any** valid timestep. We use `JUMP_Z = 0.8` (clearly above the resting stance).

### What we count
For every preference pair we classify it into one of:
* **preferred only** — the ant jumps ≥1× in the *preferred* segment but not at all in the non-preferred segment,
* **non-preferred only** — jumps ≥1× in the *non-preferred* segment but not at all in the preferred segment,
* **both** — jumps ≥1× in *both* segments.

(Pairs with no jump in either segment are reported as `neither` for completeness.)

In [ ]:
import glob
import os

import h5py
import numpy as np
import pandas as pd

# --- configuration ---
DATA_DIR = "../data/antmaze"
Z_INDEX = 2  # torso z-height in the 29-dim AntMaze observation
JUMP_Z = 0.8  # a timestep counts as 'airborne' when z > JUMP_Z

nt_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_pref_nt.hdf5")))
print(f"Found {len(nt_files)} 'nt' preference files:")
for fp in nt_files:
    print("  ", os.path.basename(fp))

In [ ]:
def segment_jumps(states, mask=None, z_index=Z_INDEX, jump_z=JUMP_Z):
    """Boolean array (n_pairs,): True if the segment is airborne at >=1 valid timestep."""
    airborne = states[..., z_index] > jump_z  # (n_pairs, T)
    if mask is not None:
        airborne = airborne & mask.astype(bool)  # ignore padded timesteps
    return airborne.any(axis=1)


def analyze(path):
    with h5py.File(path, "r") as f:
        states = f["states"][:]  # (N, T, 29) -> segment 1
        states_2 = f["states_2"][:]  # (N, T, 29) -> segment 2
        labels = f["labels"][:]  # (N, 2) one-hot
        mask = f["attn_mask"][:] if "attn_mask" in f else None
        mask_2 = f["attn_mask_2"][:] if "attn_mask_2" in f else None

    jump_1 = segment_jumps(states, mask)  # seg 1 contains a jump
    jump_2 = segment_jumps(states_2, mask_2)  # seg 2 contains a jump

    seg1_is_preferred = labels[:, 0] == 1
    pref_jump = np.where(seg1_is_preferred, jump_1, jump_2)
    nonpref_jump = np.where(seg1_is_preferred, jump_2, jump_1)

    return {
        "dataset": os.path.basename(path).replace("_pref_nt.hdf5", ""),
        "preferred_only": int(np.sum(pref_jump & ~nonpref_jump)),
        "non_preferred_only": int(np.sum(~pref_jump & nonpref_jump)),
        "both": int(np.sum(pref_jump & nonpref_jump)),
        "neither": int(np.sum(~pref_jump & ~nonpref_jump)),
        "total_pairs": int(len(labels)),
    }


results = pd.DataFrame([analyze(fp) for fp in nt_files])
results

### Summary table

* **`preferred_only`** — jump in the preferred segment only.
* **`non_preferred_only`** — jump in the non-preferred segment only.
* **`both`** — jump in both segments.

(`neither` and `total_pairs` are included for context.)

In [ ]:
cols = [
    "dataset",
    "preferred_only",
    "non_preferred_only",
    "both",
    "neither",
    "total_pairs",
]
summary = results[cols].copy()
print(f"Jump threshold: torso z-height > {JUMP_Z}\n")
print(summary.to_string(index=False))